# Lab 6: Data Governance Demo

This notebook demonstrates:
- Unity Catalog permissions (GRANT)
- Row-Level Security (RLS)
- Column-Level Security (CLS)

All security policies are tested on a separate demo table.

In [0]:
%sql

CREATE TABLE IF NOT EXISTS
dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.governance_demo
USING DELTA
AS
SELECT
    country_name,
    region,
    subregion,
    year,
    total_production
FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.agg_country_year;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT country_name) AS countries,
    COUNT(DISTINCT subregion) AS subregions
FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.governance_demo;

total_rows,countries,subregions
1146,38,5


In [0]:
%sql

CREATE TABLE IF NOT EXISTS
dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.access_roles (
    user_email STRING NOT NULL,
    role STRING NOT NULL,
    allowed_subregion STRING -- subregion that can be viewed by regional_viewer.
)
USING DELTA;

In [0]:
%sql

DESCRIBE TABLE
dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.access_roles;

col_name,data_type,comment
user_email,string,null
role,string,null
allowed_subregion,string,null


In [0]:
%sql

INSERT INTO dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.access_roles
SELECT
    current_user(),
    'analyst',
    NULL
WHERE NOT EXISTS (
    SELECT 1
    FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.access_roles
    WHERE user_email = current_user()
);


num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql

SELECT *
FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.access_roles;

user_email,role,allowed_subregion
victoriya44250@softserve.academy,analyst,null


In [0]:
%sql

SHOW GRANTS ON TABLE
dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.governance_demo;

Principal,ActionType,ObjectType,ObjectKey


Grant

In [0]:
%sql

GRANT SELECT
ON TABLE dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.governance_demo
TO `victoriya44250@softserve.academy`;

In [0]:
%sql

SHOW GRANTS ON TABLE
dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.governance_demo;

Principal,ActionType,ObjectType,ObjectKey
victoriya44250@softserve.academy,SELECT,TABLE,dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.governance_demo


In [0]:
%sql

CREATE OR REPLACE FUNCTION
dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.rls_subregion_filter(
    row_subregion STRING
)
RETURNS BOOLEAN
RETURN EXISTS (
    SELECT 1
    FROM dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.access_roles
    WHERE user_email = current_user()
      AND (
          role IN ('admin', 'analyst')
          OR (
              role = 'regional_viewer'
              AND allowed_subregion = row_subregion
          )
      )
);